# 🎤 KaraokeAI Backend Server
### Run this notebook to start the backend API for your KaraokeAI app
**Steps:** Click **Runtime → Run all** after setting your ngrok token in Cell 4.
- After Cell 5 runs, you will get a **public URL** — copy and paste it into your app!

In [8]:
# ── CELL 1: Install all dependencies ──────────────────────────
print('Installing dependencies... (takes 2-3 minutes)')
!pip install -q faster-whisper demucs ffmpeg-python pysubs2 fastapi uvicorn pyngrok nest-asyncio transliterate anyascii aksharamukha
!apt-get install -q ffmpeg
print('✅ All dependencies installed!')

Installing dependencies... (takes 2-3 minutes)
Reading package lists...
Building dependency tree...
Reading state information...
ffmpeg is already the newest version (7:6.1.1-3ubuntu5).
0 upgraded, 0 newly installed, 0 to remove and 52 not upgraded.
✅ All dependencies installed!


In [9]:
# ── CELL 2: Clone your GitHub repository ──────────────────────
import os

GITHUB_REPO = 'https://github.com/shanmuganathan457/karaoke-video-generator.git'

if os.path.exists('karaoke-video-generator'):
    print('Repo already exists, pulling latest changes...')
    os.system('cd karaoke-video-generator && git pull')
else:
    print('Cloning your GitHub repository...')
    os.system(f'git clone {GITHUB_REPO}')

os.chdir('karaoke-video-generator')
print(f'✅ Working directory: {os.getcwd()}')
print('📁 Files found:', os.listdir('.'))

Cloning your GitHub repository...
✅ Working directory: /content/karaoke-video-generator/karaoke-video-generator
📁 Files found: ['vite.config.ts', 'generate_test.ps1', 'src', '.gitignore', 'KaraokeAI_Backend.ipynb', 'mobile-app', 'inputs', 'package-lock.json', 'tsconfig.json', 'launch-page', 'README.md', 'outputs', '.env.example', 'eas.json', '.git', 'app.json', 'metadata.json', 'lyrics', 'scratch_natpe.py', 'hf_space', 'package.json', 'karaoke_generator', 'scratch', 'vercel.json', 'index.html', '.expo']


In [10]:
# ── CELL 3: Write the FastAPI server (with Monkey Patch fix) ──
api_code = '''
import sys, os, uuid
sys.path.insert(0, os.getcwd())

# --- MONKEY PATCH: Fix for av library bug on Python 3.13 ---
import av
_orig_av_open = av.open
def _patched_av_open(*args, **kwargs):
    kwargs.pop("metadata_errors", None)
    return _orig_av_open(*args, **kwargs)
av.open = _patched_av_open
# -----------------------------------------------------------

from fastapi import FastAPI, UploadFile, File, Form
from fastapi.responses import FileResponse, JSONResponse
from fastapi.middleware.cors import CORSMiddleware
from karaoke_generator.transcriber import AudioTranscriber
from karaoke_generator.subtitle_generator import SubtitleGenerator
from karaoke_generator.video_processor import VideoProcessor
from karaoke_generator.vocal_separator import VocalSeparator

app = FastAPI(title="KaraokeAI Backend API")
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)

@app.get("/")
def root():
    return {"status": "KaraokeAI Backend is running!", "version": "1.0.0"}

@app.post("/generate")
async def generate_karaoke(
    video: UploadFile = File(...),
    language: str = Form(default=""),
    romanize: bool = Form(default=True)
):
    job_id = str(uuid.uuid4())[:8]
    input_path = f"/tmp/input_{job_id}.mp4"
    temp_audio = f"/tmp/audio_{job_id}.wav"
    output_ass  = f"/tmp/karaoke_{job_id}.ass"
    output_video = f"/tmp/output_{job_id}.mp4"
    vocals_audio = None
    instrumental_audio = None

    try:
        with open(input_path, "wb") as f:
            f.write(await video.read())

        processor = VideoProcessor()
        transcriber = AudioTranscriber()
        sub_gen = SubtitleGenerator()

        processor.extract_audio(input_path, temp_audio)
        vocals_audio, instrumental_audio = VocalSeparator.separate(temp_audio)
        result = transcriber.transcribe(
            vocals_audio,
            language=language if language else None,
            vad_filter=False
        )
        sub_gen.generate(result["segments"], output_ass, romanize=romanize, language=result["language"])
        processor.burn_subtitles(input_path, output_ass, output_video)

        return FileResponse(output_video, media_type="video/mp4", filename="karaoke_output.mp4")

    except Exception as e:
        return JSONResponse(status_code=500, content={"error": str(e)})

    finally:
        for f in [input_path, temp_audio, vocals_audio, instrumental_audio, output_ass]:
            if f and os.path.exists(f):
                os.remove(f)
'''

with open('api_server.py', 'w') as f:
    f.write(api_code)

print('✅ API server created with Monkey Patch fix!')

✅ API server created with Monkey Patch fix!


In [13]:
# ── CELL 4: Enter your ngrok token ────────────────────────────
# Get your FREE token from: https://dashboard.ngrok.com/get-started/your-authtoken
# Paste your token below between the quotes:

NGROK_TOKEN = "3K3ZdIf4fOiZoKNgHMNkp1OMjzx_4m3KxC1tiARvKtNiZKKq7"

from pyngrok import ngrok
ngrok.set_auth_token(NGROK_TOKEN)
print('✅ ngrok token set!')

✅ ngrok token set!


In [15]:
# ── CELL 5: START THE SERVER ──────────────────────────────────
# This prints your public URL — copy it and paste into your React app!

import subprocess
import time
from pyngrok import ngrok

# Kill any existing tunnels
ngrok.kill()

# Open a public tunnel on port 8000
public_url = ngrok.connect(8000)

print('=' * 60)
print('🚀 KaraokeAI Backend is LIVE!')
print('=' * 60)
print(f'\n📡 YOUR PUBLIC API URL:')
print(f'   {public_url}')
print(f'\n✅ Copy this URL and give it to your developer!')
print(f'\n⚠️  Keep this tab open — closing it stops the server.')
print('=' * 60)

# Start FastAPI server in background (avoids Colab asyncio conflict)
subprocess.Popen(
    ["uvicorn", "api_server:app", "--host", "0.0.0.0", "--port", "8000"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

# Keep cell alive
while True:
    time.sleep(60)
    print('💓 Server still running...')

🚀 KaraokeAI Backend is LIVE!

📡 YOUR PUBLIC API URL:
   NgrokTunnel: "https://defiance-trustable-washstand.ngrok-free.dev" -> "http://localhost:8000"

✅ Copy this URL and give it to your developer!

⚠️  Keep this tab open — closing it stops the server.
💓 Server still running...
💓 Server still running...
💓 Server still running...
💓 Server still running...
💓 Server still running...
💓 Server still running...
💓 Server still running...
💓 Server still running...
💓 Server still running...
💓 Server still running...
💓 Server still running...
💓 Server still running...
💓 Server still running...
💓 Server still running...
💓 Server still running...
💓 Server still running...
💓 Server still running...
💓 Server still running...
💓 Server still running...
💓 Server still running...
💓 Server still running...
💓 Server still running...
💓 Server still running...
💓 Server still running...
💓 Server still running...
💓 Server still running...
💓 Server still running...
💓 Server still running...
💓 Server still runn

KeyboardInterrupt: 